# Code-Gen Pipeline — Prompt Inspector

Compile and inspect the assembled prompt for every code-gen stage.

| Stage | What it builds |
|---|---|
| state1 | Entity list (JSON) |
| state1b | Policy outline (JSON) |
| state1c | Entity dependency DAG (JSON) |
| state1d | Metrics draft (JSON) |
| state2 | Entity class code (Python) |
| state3 | Environment class code (Python) |
| state4 | Policy class code (Python) |
| state5 | Policy judge (LLM-as-judge) |

In [ ]:
import sys, pathlib, json, importlib.util
from IPython.display import display, Markdown

# ── path setup ──────────────────────────────────────────────────────────────
REPO    = pathlib.Path.cwd().parents[1]          # Garbage-Management-Simulation-Engine/
BACKEND = REPO / "engine" / "backend"
_SVC    = BACKEND / "app" / "services"

# Direct module load — avoids triggering app/__init__.py → FastAPI import
def _load_module(name: str, path: pathlib.Path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

P = _load_module("code_gen_prompts", _SVC / "code_gen_prompts.py")

# ── display helper ──────────────────────────────────────────────────────────
def show(title: str, text: str, lang: str = "text", max_lines: int = 0) -> None:
    lines = text.splitlines()
    if max_lines and len(lines) > max_lines:
        body = "\n".join(lines[:max_lines]) + f"\n\n… ({len(lines) - max_lines} more lines)"
    else:
        body = text
    display(Markdown(f"### {title}\n\n```{lang}\n{body}\n```"))

def show_json(title: str, obj, max_lines: int = 0) -> None:
    show(title, json.dumps(obj, indent=2, ensure_ascii=False), lang="json", max_lines=max_lines)

print("Setup OK — backend path:", BACKEND)

---
## Fixtures
Minimal stub data used to assemble every prompt below.  
Replace with real causal data to see production-equivalent output.

In [ ]:
CAUSAL_DATA = json.dumps([
    {
        "sentence": "Waste bins are placed throughout the building. When a bin is full, a janitor empties it into a collection truck.",
        "classes": [
            {"id": "waste_bin", "label": "Waste Bin", "sentence_type": "CS"},
            {"id": "janitor",   "label": "Janitor",   "sentence_type": "CS"},
            {"id": "truck",     "label": "Collection Truck", "sentence_type": "CS"},
        ]
    }
], ensure_ascii=False)

ENTITIES = [
    {"id": "waste_bin", "label": "Waste Bin",        "type": "resource", "frequency": 5},
    {"id": "janitor",   "label": "Janitor",           "type": "actor",    "frequency": 3},
    {"id": "truck",     "label": "Collection Truck",  "type": "resource", "frequency": 2},
]

POLICIES = [
    {
        "rule_id": "empty_full_bin",
        "label": "Empty Full Bin",
        "trigger": "waste_bin.fill_level >= capacity",
        "target_entity_id": "waste_bin",
        "target_method": "empty",
        "inputs": ["janitor"],
        "description": "Janitor empties a full waste bin into the truck.",
    }
]

EDGES = [
    {"from": "janitor", "to": "waste_bin", "reason": "janitor calls waste_bin.empty()"},
    {"from": "janitor", "to": "truck",     "reason": "janitor deposits waste into truck"},
]

METRICS = [
    {
        "name": "bin_fill_level",
        "label": "Bin Fill Level",
        "unit": "%",
        "agg": "mean",
        "viz": "line",
        "entity_id": "waste_bin",
        "expected_variable": "fill_level",
        "chart_type": "line",
        "how_to_interpret": "Average fill level across all bins per tick.",
        "required_attrs": [{"entity": "waste_bin", "attr": "fill_level"}],
        "sampling_event": "tick",
        "rationale": "Core operational metric for overflow prevention.",
    }
]

SAMPLE_ENTITY_OBJ = ENTITIES[0]   # waste_bin — used in state2 demo
SAMPLE_RULE       = POLICIES[0]   # empty_full_bin — used in state4 demo

print("Fixtures ready.")

---
## Code Templates
Raw base-class files injected into State 2 / 3 / 4 prompts.

In [ ]:
TEMPLATE_DIR = BACKEND / "app" / "services" / "templates"

TMPL_ENTITY  = (TEMPLATE_DIR / "entity_object_template.py").read_text()
TMPL_ENV     = (TEMPLATE_DIR / "environment_template.py").read_text()
TMPL_POLICY  = (TEMPLATE_DIR / "policy_template.py").read_text()

templates = {
    "entity_object_template.py":  TMPL_ENTITY,
    "environment_template.py":    TMPL_ENV,
    "policy_template.py":         TMPL_POLICY,
}

for name, src in templates.items():
    show(name, src, lang="python", max_lines=40)

---
## State 1 — Entity List
**Input**: causal data  
**Output schema**: `{entities: [{id, label, type, frequency}]}`

In [ ]:
prompt_s1, schema_s1 = P.build_state1_entity_list_prompt(CAUSAL_DATA)

show_json("State 1 — Response Schema", schema_s1)
show("State 1 — Assembled Prompt", prompt_s1)

---
## State 1b — Policy Outline
**Input**: causal data + entity list  
**Output schema**: `{policies: [{rule_id, trigger, target_entity_id, target_method, ...}]}`

In [ ]:
prompt_s1b, schema_s1b = P.build_state1b_policy_outline_prompt(CAUSAL_DATA, ENTITIES)

show_json("State 1b — Response Schema", schema_s1b)
show("State 1b — Assembled Prompt", prompt_s1b)

---
## State 1c — Entity Dependencies
**Input**: causal data + entity list + policy outline  
**Output schema**: `{edges: [{from, to, reason}]}`

In [ ]:
prompt_s1c, schema_s1c = P.build_state1c_entity_dependencies_prompt(
    CAUSAL_DATA, ENTITIES, POLICIES
)

show_json("State 1c — Response Schema", schema_s1c)
show("State 1c — Assembled Prompt", prompt_s1c)

---
## State 1d — Metrics Draft
**Input**: entity list + policy outline + dependency edges  
**Output schema**: `{metrics: [{name, label, entity_id, expected_variable, ...}]}`

In [ ]:
prompt_s1d, schema_s1d = P.build_state1d_metrics_draft_prompt(
    entities=ENTITIES,
    policy_outline=POLICIES,
    dependency_edges=EDGES,
)

show_json("State 1d — Response Schema", schema_s1d)
show("State 1d — Assembled Prompt", prompt_s1d)

---
## State 2 — Entity Object Code
**Template injected**: `entity_object_template.py`  
**Iterative**: one call per entity in topological order  
**Output**: Python class subclassing `entity_object`

Two sub-cells:
1. Stable cached context (shared across all iterations)
2. Per-entity variable prompt (for one sample entity)

In [ ]:
# ── 2a: Cached context (sent once, reused for all entities) ─────────────────
cached_parts = P.build_state2_cached_context(causal_data=CAUSAL_DATA)
show(
    "State 2 — Cached Context (stable prefix, shared across all entity iterations)",
    "\n\n".join(cached_parts),
    max_lines=60,
)

In [ ]:
# ── 2b: Per-entity prompt (sample: waste_bin) ────────────────────────────────
topo_order = P.topological_order(ENTITIES, EDGES)
print("Topological order:", topo_order)

prompt_s2 = P.build_state2_entity_prompt(
    causal_data=CAUSAL_DATA,
    entity_id="waste_bin",
    entity_obj=SAMPLE_ENTITY_OBJ,
    accumulator_blob="",           # empty — first entity
    interface_digest={"classes": []},
    policy_outline=POLICIES,
    selected_metrics=METRICS,
    omit_cached_context=False,     # inline mode (no Gemini cache)
)

show("State 2 — Assembled Prompt (entity: waste_bin, inline mode)", prompt_s2)

In [ ]:
# ── 2c: Same prompt with omit_cached_context=True (cache-hit mode) ───────────
prompt_s2_cached = P.build_state2_entity_prompt(
    causal_data=CAUSAL_DATA,
    entity_id="waste_bin",
    entity_obj=SAMPLE_ENTITY_OBJ,
    accumulator_blob="",
    interface_digest={"classes": []},
    policy_outline=POLICIES,
    selected_metrics=METRICS,
    omit_cached_context=True,      # stable prefix supplied via Gemini cache
)

show("State 2 — Assembled Prompt (entity: waste_bin, cache-hit mode)", prompt_s2_cached)

---
## State 3 — Environment Code
**Template injected**: `environment_template.py`  
**Input**: all generated entity code (accumulator blob) + policy outline + map graph  
**Output**: Python class subclassing `SimulationEnvironment`

In [ ]:
# Fake accumulator blob: stub entity code representing prior state2 output
STUB_ENTITY_CODE = """\
# === FILE: waste_bin.py ===
from entity_object_template import entity_object

class Entity_WasteBin(entity_object):
    def __init__(self, entity_object_id: str):
        super().__init__(entity_object_id)
        self.fill_level: float = 0.0
        self.capacity: float = 100.0

    def step(self, dt: float, env) -> None:
        pass

    def empty(self) -> None:
        self.fill_level = 0.0

    def on_query(self, metric_name: str) -> dict:
        if metric_name == 'bin_fill_level':
            return {'fill_level': self.fill_level}
        return {}
"""

SAMPLE_MAP_GRAPH = {
    "vertices": [
        {"id": "building_a", "label": "Building A", "type": "building", "x": 0, "y": 0},
        {"id": "depot",      "label": "Waste Depot",  "type": "depot",    "x": 100, "y": 0},
    ],
    "edges": [
        {"id": "e1", "source": "building_a", "target": "depot", "label": "road", "weight": 1.0}
    ]
}

prompt_s3 = P.build_state3_environment_prompt(
    entities_blob=STUB_ENTITY_CODE,
    policy_outline=POLICIES,
    map_graph=SAMPLE_MAP_GRAPH,
)

show("State 3 — Assembled Prompt (with map graph)", prompt_s3)

In [ ]:
# ── State 3 without map (map_graph=None) ─────────────────────────────────────
prompt_s3_nomap = P.build_state3_environment_prompt(
    entities_blob=STUB_ENTITY_CODE,
    policy_outline=POLICIES,
    map_graph=None,
)

show("State 3 — Assembled Prompt (no map)", prompt_s3_nomap)

---
## State 4 — Policy Code
**Template injected**: `policy_template.py`  
**Iterative**: one call per policy rule  
**Output**: Python class subclassing `Policy`

In [ ]:
STUB_ENV_CODE = """\
from environment_template import SimulationEnvironment

class GarbageSimulationEnvironment(SimulationEnvironment):
    def __init__(self, entities=None, policies=None):
        super().__init__(entities=entities, policies=policies)
"""

prompt_s4 = P.build_state4_policy_prompt(
    causal_data=CAUSAL_DATA,
    rule=SAMPLE_RULE,
    entities_blob=STUB_ENTITY_CODE,
    environment_code=STUB_ENV_CODE,
    policies_accumulator="",       # empty — first policy
    map_graph=SAMPLE_MAP_GRAPH,
)

show("State 4 — Assembled Prompt (rule: empty_full_bin)", prompt_s4)

---
## State 5 — Policy Judge (Pass 1 + Pass 2)
**LLM-as-Judge**: reviews generated policy for concrete bugs.  
Pass 1 → identify issues.  Pass 2 → fix them.

In [ ]:
SAMPLE_POLICY_CODE = """\
from policy_template import Policy

class Entity_EmptyFullBinPolicy(Policy):
    @property
    def policy_name(self) -> str:
        return 'empty_full_bin'

    def apply(self, entity_object, context, env=None):
        return {}

    def before_tick(self, env, dt):
        for entity in env.entities:
            if hasattr(entity, 'fill_level') and entity.fill_level >= entity.capacity:
                entity.emtpy()   # intentional typo — judge should catch this
"""

ENTITY_CODE_INDEX = {"waste_bin": STUB_ENTITY_CODE}

MAP_ACCESSOR_API = """\
env.get_nodes() -> list[dict]
env.get_node(node_id) -> dict | None
env.get_edges() -> list[dict]
env.get_neighbors(node_id) -> list[str]
env.get_node_types() -> list[str]"""

# ── Pass 1: identify bugs ────────────────────────────────────────────────────
prompt_judge_p1 = P.build_policy_judge_pass1_prompt(
    policy_code=SAMPLE_POLICY_CODE,
    rule_contract=SAMPLE_RULE,
    entity_code_index=ENTITY_CODE_INDEX,
    map_accessor_api=MAP_ACCESSOR_API,
)
show("State 5 — Judge Pass 1 Prompt (identify bugs)", prompt_judge_p1)

In [ ]:
# ── Pass 2: fix bugs ─────────────────────────────────────────────────────────
SAMPLE_ISSUES = [
    {
        "severity": "critical",
        "location": "before_tick",
        "description": "entity.emtpy() is a typo — method is entity.empty()",
        "suggested_fix": "Change entity.emtpy() to entity.empty()",
    }
]

prompt_judge_p2 = P.build_policy_judge_pass2_prompt(
    policy_code=SAMPLE_POLICY_CODE,
    rule_contract=SAMPLE_RULE,
    entity_code_index=ENTITY_CODE_INDEX,
    map_accessor_api=MAP_ACCESSOR_API,
    issues=SAMPLE_ISSUES,
)
show("State 5 — Judge Pass 2 Prompt (fix bugs)", prompt_judge_p2)

---
## Protocol Validators
Dry-run the validators against stub code to verify they work as expected.

In [ ]:
GOOD_ENTITY = """\
from entity_object_template import entity_object
class Entity_WasteBin(entity_object):
    def step(self, dt: float, env) -> None:
        pass
    def empty(self) -> None:
        self.fill_level = 0.0
"""
BAD_ENTITY = GOOD_ENTITY.replace("def step", "def _step")  # missing step

GOOD_ENV = """\
from environment_template import SimulationEnvironment
class MyEnv(SimulationEnvironment):
    def __init__(self, entities=None, policies=None):
        super().__init__(entities, policies)
    def tick(self, dt: float) -> None:
        super().tick(dt)
"""

GOOD_POLICY = """\
from policy_template import Policy
class EmptyBinPolicy(Policy):
    @property
    def policy_name(self): return 'empty_bin'
    def apply(self, entity_object, context, env=None): return {}
    def before_tick(self, env, dt):
        pass
"""

cases = [
    ("Entity GOOD",  P.validate_entity_protocol,      GOOD_ENTITY,  ["empty"]),
    ("Entity BAD",   P.validate_entity_protocol,      BAD_ENTITY,   ["empty"]),
    ("Env GOOD",     P.validate_environment_protocol, GOOD_ENV,     None),
    ("Policy GOOD",  P.validate_policy_protocol,      GOOD_POLICY,  None),
]

for label, fn, src, extra in cases:
    if extra is not None:
        errs = fn(src, required_methods=extra)
    else:
        errs = fn(src)
    status = "✅ PASS" if not errs else f"❌ FAIL: {errs}"
    print(f"{label:20s} → {status}")

---
## Prompt Size Summary
Character counts for every assembled prompt — useful for estimating token usage.

In [ ]:
named_prompts = [
    ("state1  entity list",          prompt_s1),
    ("state1b policy outline",        prompt_s1b),
    ("state1c entity deps",           prompt_s1c),
    ("state1d metrics draft",         prompt_s1d),
    ("state2  entity (inline)",       prompt_s2),
    ("state2  entity (cache-hit)",    prompt_s2_cached),
    ("state2  cached context",        "\n\n".join(cached_parts)),
    ("state3  env (with map)",        prompt_s3),
    ("state3  env (no map)",          prompt_s3_nomap),
    ("state4  policy",                prompt_s4),
    ("state5  judge pass1",           prompt_judge_p1),
    ("state5  judge pass2",           prompt_judge_p2),
]

print(f"{'Stage':<32} {'chars':>8}  {'~tokens':>8}")
print("-" * 54)
for name, text in named_prompts:
    chars = len(text)
    approx_tokens = chars // 4
    print(f"{name:<32} {chars:>8,}  {approx_tokens:>8,}")